In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## Gradio ve Chromadb Kütüphanesini Yükle

In [2]:
!pip install -q chromadb


import chromadb


%pip install -q gradio


# Langchain kütüphanesini yükle (eğer yüklü değilse)
%pip install -q langchain


%pip install -q PyPDF2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 73.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

## Bilgi Tabanı (Knowledge Base) Oluştur

### Örnek Belgeleri İndir

In [3]:
# @title Örnek Belgeleri İndir
import requests
from urllib.parse import quote

# GitHub repository bilgileri
owner = "zemzemyayan"
repo = "rag-recipe-assistant"
folder_path = "data"

# API endpoint
api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{quote(folder_path)}"

print("GitHub API'ye bağlanılıyor...")
try:
    response = requests.get(api_url)
    response.raise_for_status()

    files = response.json()
    pdf_files = [f for f in files if f['name'].lower().endswith('.pdf')]

    print(f"{len(pdf_files)} PDF dosyası bulundu:")

    downloaded_files = []

    for file in pdf_files:
        print(f"\nİndiriliyor: {file['name']}")

        try:
            file_response = requests.get(file['download_url'])
            file_response.raise_for_status()

            file_path = f"./{file['name']}"

            with open(file_path, 'wb') as f:
                f.write(file_response.content)

            file_size = len(file_response.content)
            print(f"✓ {file['name']} indirildi ({file_size:,} bytes)")
            print(f"  Konum: {file_path}")
            downloaded_files.append(file_path)

        except Exception as e:
            print(f"✗ {file['name']} indirilemedi: {e}")

    print(f"\n🎉 Toplam {len(downloaded_files)} PDF dosyası indirildi!")
    print("\nİndirilen dosyalar:")
    for file_path in downloaded_files:
        print(f"📄 {file_path}")

except Exception as e:
    print(f"API hatası: {e}")
    import traceback
    traceback.print_exc()

GitHub API'ye bağlanılıyor...
1 PDF dosyası bulundu:

İndiriliyor: yemek_tarifleri_rehberi.pdf
✓ yemek_tarifleri_rehberi.pdf indirildi (54,431 bytes)
  Konum: ./yemek_tarifleri_rehberi.pdf

🎉 Toplam 1 PDF dosyası indirildi!

İndirilen dosyalar:
📄 ./yemek_tarifleri_rehberi.pdf


### PDF Belgelerini Yükle

In [4]:
# create_knowledge_base() fonksiyonu, mevcut dizindeki tüm PDF dosyalarını
# bulup listeleyerek dosya yollarını bir liste halinde döndürür.

import os

def create_knowledge_base():
  # İndirilen dosya yollarını tutacak boş bir liste oluşturun
  downloaded_files = []

  # Mevcut dizindeki tüm dosya ve klasörleri listele
  for item in os.listdir('.'):

      # Eğer öğe bir dosyaysa ve '.pdf' ile bitiyorsa
      if os.path.isfile(item) and item.lower().endswith('.pdf'):

          # Dosya yolunu downloaded_files listesine ekle
          downloaded_files.append(item)

  # Bulunan PDF dosyalarını yazdır
  print("Yerel dizinde bulunan PDF dosyaları:")

  for file_path in downloaded_files:
      print(f"📄 {file_path}")

  print(f"\nToplam {len(downloaded_files)} PDF dosyası bulundu.")
  return downloaded_files

In [5]:
liste=create_knowledge_base()
liste

Yerel dizinde bulunan PDF dosyaları:
📄 yemek_tarifleri_rehberi.pdf

Toplam 1 PDF dosyası bulundu.


['yemek_tarifleri_rehberi.pdf']

## Belgelerden Metin Çıkarımı (Text Extraction):

In [6]:
import PyPDF2

In [7]:
# # convert_PDF_Text() fonksiyonu, verilen PDF dosyasındaki sayfalardan metinleri çıkarıp
# boş metinleri filtreleyerek sayfa metinlerini liste halinde döndürür.

def convert_PDF_Text(pdf_path):

  try:

    # Belirtilen PDF dosyasını okumak için PdfReader nesnesi oluştur
    reader = PyPDF2.PdfReader(pdf_path)

    # Her sayfadan metni çıkar ve baş/son boşlukları temizle
    pdf_pages = [p.extract_text().strip() for p in reader.pages]

    # Boş metin dizilerini filtrele
    pdf_pages = [text for text in pdf_pages if text]

    # Belge adını ve işlenen sayfa sayısını yazdır
    print("Belge: ", pdf_path,"\nSayfa Sayısı: ", len(pdf_pages))

    # Metin sayfalarının listesini döndür
    return pdf_pages

  except Exception as e:
    print(f"Hata oluştu: {pdf_path} dosyası işlenemedi. Hata: {e}")

    return [] # Hata durumunda boş liste döndür

In [8]:
pdfText=convert_PDF_Text('yemek_tarifleri_rehberi.pdf')
# pdfText=convert_PDF_Text(liste[0])

pdfText

Belge:  yemek_tarifleri_rehberi.pdf 
Sayfa Sayısı:  15


['Geleneksel Türk Mutfağı Yemek Tarifleri Rehberi\nÇorbalardan Ana Yemeklere, Zeytinyağlılardan Tatlılara 20 Otantik Tarif\nBu rehber, Anadolu mutfak kültürünün asırlık geleneklerinden süzülerek günümüze ulaşmış en bilinen ve\nsevilen tariflerini bir araya getirmektedir. Her bir tarif; malzeme listesi, aşama aşama hazırlama talimatları ve\npüf noktaları ile eksiksiz olarak sunulmuştur. Bildirim ve sunum ögelerinden arındırılmış, tamamen yazılı\nanlatıma dayalı bu çalışma, mutfakta uygulamalı bir kaynak olarak tasarlanmıştır.\nBÖLÜM 1: ÇORBALAR\n1. Geleneksel Mercimek Çorbası\nAnadolu mutfağının en köklü ve yaygın çorbalarından biri olan kırmızı mercimek çorbası, hem besleyici\ndeğeri hem de yumuşak içimiyle sofraların vazgeçilmezidir.\nMalzemeler:\n1.5 su bardağı kırmızı mercimek (iyice yıkanmış ve süzülmüş)\n1 adet orta boy kuru soğan (yemeklik doğranmış)\n1 adet orta boy patates (küp doğranmış)\n1 adet orta boy havuç (küp doğranmış)\n1 yemek kaşığı un\n1 yemek kaşığı tereyağı\n2 yeme

## Metin İşleme (Chunking)

In [9]:
# Langchain kütüphanesini yükle (eğer yüklü değilse)

!pip install -q langchain-text-splitters

In [10]:
# Metinleri parçalara ayırmak için gerekli sınıfları içe aktar

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    SentenceTransformersTokenTextSplitter,
)

In [11]:
#  convert_Page_ChunkinChar() fonksiyonu, PDF'den çıkarılan metinleri belirlenen karakter boyutu ve örtüşme miktarına göre
# daha küçük parçalara (chunk) ayırarak liste halinde döndürür.

def convert_Page_ChunkinChar(pdf_texts, chunk_size = 1500, chunk_overlap=200 ):

  # Karakter tabanlı metin ayırıcı nesnesi oluştur
  character_splitter = RecursiveCharacterTextSplitter(

      # Ayırıcı karakterleri tanımla
      separators=["\n\n", "\n", ". ", " ", ""],

      # Her parçanın maksimum karakter boyutunu belirle
      chunk_size=chunk_size,

      # Parçalar arasındaki çakışma miktarını belirle
      chunk_overlap=chunk_overlap
  )

  # PDF metinlerini belirtilen ayırıcılar ve boyutlarla parçalara ayır
  character_split_texts = character_splitter.split_text('\n\n'.join(pdf_texts))

  # Toplam parça sayısını yazdır
  print(f"\nToplam parça sayısı (belge maksimum karakter boyutuna göre bölündü = {chunk_size}): \
        {len(character_split_texts)}")

  # Oluşturulan metin parçalarının listesini döndür
  return character_split_texts

In [12]:
chunksTextList=convert_Page_ChunkinChar(pdfText, 1500, 200)
print(len(chunksTextList))
for i in range(5):
  print(chunksTextList[i])


Toplam parça sayısı (belge maksimum karakter boyutuna göre bölündü = 1500):         22
22
Geleneksel Türk Mutfağı Yemek Tarifleri Rehberi
Çorbalardan Ana Yemeklere, Zeytinyağlılardan Tatlılara 20 Otantik Tarif
Bu rehber, Anadolu mutfak kültürünün asırlık geleneklerinden süzülerek günümüze ulaşmış en bilinen ve
sevilen tariflerini bir araya getirmektedir. Her bir tarif; malzeme listesi, aşama aşama hazırlama talimatları ve
püf noktaları ile eksiksiz olarak sunulmuştur. Bildirim ve sunum ögelerinden arındırılmış, tamamen yazılı
anlatıma dayalı bu çalışma, mutfakta uygulamalı bir kaynak olarak tasarlanmıştır.
BÖLÜM 1: ÇORBALAR
1. Geleneksel Mercimek Çorbası
Anadolu mutfağının en köklü ve yaygın çorbalarından biri olan kırmızı mercimek çorbası, hem besleyici
değeri hem de yumuşak içimiyle sofraların vazgeçilmezidir.
Malzemeler:
1.5 su bardağı kırmızı mercimek (iyice yıkanmış ve süzülmüş)
1 adet orta boy kuru soğan (yemeklik doğranmış)
1 adet orta boy patates (küp doğranmış)
1 adet orta bo

In [13]:
# Metinleri token bazlı parçalara ayırmak için gerekli sınıfı içe aktar
# Kullanılacak Sentence Transformer modelinin adını tanımla
# Bu model, metin parçalarını tokenlara ayırmak ve embedding işlemi için kullanılacak.
sentence_transformer_model="distiluse-base-multilingual-cased-v1"


### Metinleri token bazlı parçalara ayıran yardımcı fonksiyon

In [14]:
# # convert_Chunk_Token() fonksiyonu, karakter bazlı chunk'ları seçilen tokenizer modeliyle belirlenen token sayısı ve
# örtüşme miktarına göre token tabanlı daha küçük parçalara ayırarak liste halinde döndürür.

def convert_Chunk_Token(text_chunksinChar,sentence_transformer_model, chunk_overlap=10,tokens_per_chunk=128 ):

  # SentenceTransformersTokenTextSplitter nesnesi oluşturuluyor
  token_splitter = SentenceTransformersTokenTextSplitter(

      chunk_overlap=chunk_overlap, # Parçalar arası çakışma miktarı

      model_name=sentence_transformer_model, # Kullanılacak tokenizer modelinin adı

      tokens_per_chunk=tokens_per_chunk) # Her bir parçadaki maksimum token sayısı

  # Tokenlara ayrılmış metin parçalarını tutacak boş liste
  text_chunksinTokens = []

  # Karakter bazlı parçalar üzerinde döngü
  for text in text_chunksinChar:

      # Her bir karakter bazlı parçayı tokenlara ayır ve listeye ekle
      text_chunksinTokens += token_splitter.split_text(text)

  # Bilgilendirme mesajı yazdır
  print(f"""\nBelge {tokens_per_chunk} token'lık parçalara bölündüğünde,
  ve çakışma token sayısı {chunk_overlap} olduğunda
  toplam parça sayısı: {len(text_chunksinTokens)}""")

  # Tokenlara ayrılmış metin parçaları listesini döndür
  return text_chunksinTokens


In [40]:
chunksTokenLİst=convert_Chunk_Token(chunksTextList,sentence_transformer_model, chunk_overlap=10,tokens_per_chunk=128 )
print(len(chunksTokenLİst))
for i in range(5):
  print(chunksTokenLİst[i])
  print("\n")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]


Belge 128 token'lık parçalara bölündüğünde,
  ve çakışma token sayısı 10 olduğunda
  toplam parça sayısı: 79
79
Geleneksel Türk Mutfağı Yemek Tarifleri Rehberi Çorbalardan Ana Yemeklere, Zeytinyağlılardan Tatlılara 20 Otantik Tarif Bu rehber, Anadolu mutfak kültürünün asırlık geleneklerinden süzülerek günümüze ulaşmış en bilinen ve sevilen tariflerini bir araya getirmektedir. Her bir tarif ; malzeme listesi, aşama aşama hazırlama talimatları ve püf noktaları ile eksiksiz olarak sunulmuştur. Bildirim ve


##z olarak sunulmuştur. Bildirim ve sunum ögelerinden arındırılmış, tamamen yazılı anlatıma dayalı bu çalışma, mutfakta uygulamalı bir kaynak olarak tasarlanmıştır. BÖLÜM 1 : ÇORBALAR 1. Geleneksel Mercimek Çorbası Anadolu mutfağının en köklü ve yaygın çorbalarından biri olan kırmızı mercimek çorbası, hem besleyici değeri hem de yumuşak içimiyle sofraların vazgeçi


içimiyle sofraların vazgeçilmezidir. Malzemeler : 1. 5 su bardağı kırmızı mercimek ( iyice yıkanmış ve süzülmüş ) 1 adet

## Vektör Veritabanı Hazırla

In [16]:
vector_database_path = "./" # Mevcut dizini kullan

In [17]:
from chromadb.utils import embedding_functions
# SentenceTransformer modelini kullanarak bir embedding fonksiyonu oluştur
# Bu fonksiyon, metin parçalarını vektörlere dönüştürmek için kullanılacak

# önce embedding modelini seç
sentence_transformer_model="distiluse-base-multilingual-cased-v1"

# sonra fonksiyonu oluştur
embedding_function= embedding_functions.SentenceTransformerEmbeddingFunction(model_name=sentence_transformer_model)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [18]:
# create_chroma_client() fonksiyonu, belirtilen konumda kalıcı bir ChromaDB istemcisi ve koleksiyonu oluşturur;
# koleksiyon zaten varsa silip yeniden oluşturur.

def create_chroma_client(vector_database_path, collection_name, embedding_function):
  # Kalıcı (Persistent) bir ChromaDB istemcisi oluştur

  # Bu, vektör veritabanının diske kaydedilmesini ve oturumlar arasında kalıcı olmasını sağlar
  chroma_client = chromadb.PersistentClient(path=vector_database_path)

  # Belirtilen isimde bir koleksiyonun var olup olmadığını kontrol et
  try:
      chroma_collection = chroma_client.get_collection(collection_name)
      print(f"'{collection_name}' adlı koleksiyon zaten mevcut. Siliniyor...")

      # Eğer koleksiyon mevcutsa sil
      chroma_client.delete_collection(collection_name)
      print(f"'{collection_name}' adlı koleksiyon silindi.")

      # Silme işleminden sonra koleksiyonu yeniden oluştur
      chroma_collection = chroma_client.create_collection(collection_name, embedding_function=embedding_function)
      print(f"'{collection_name}' adlı koleksiyon yeniden oluşturuldu.")

  except:
      # Eğer koleksiyon mevcut değilse, doğrudan oluştur
      chroma_collection = chroma_client.create_collection(collection_name, embedding_function=embedding_function)
      print(f"'{collection_name}' adlı yeni koleksiyon oluşturuldu.")


  # Oluşturulan istemciyi ve koleksiyonu döndür
  return chroma_client, chroma_collection

In [19]:
# # add_meta_data() fonksiyonu, metin parçaları için benzersiz ID'ler oluşturarak her parçaya
# belge adı ve kategori bilgilerini metadata olarak ekler ve bunları listeler halinde döndürür.

def add_meta_data(text_chunksinTokens, title, category, initial_id):
  # Parçalar için benzersiz kimlikler (ID'ler) oluştur

  ids = [str(i+initial_id) for i in range(len(text_chunksinTokens))]
  # Parçalara eklenecek metadata (üst veri) sözlüğü oluştur

  metadata = {
      'document': title,  # Belge adı
      'category': category # Belge kategorisi
  }

  # Her parça için aynı metadata'yı içeren bir liste oluştur
  metadatas = [ metadata for i in range(len(text_chunksinTokens))]

  # Oluşturulan ID'ler ve metadata listelerini döndür
  return ids, metadatas


In [20]:
# # add_document_to_collection() fonksiyonu, metin parçalarını benzersiz ID ve metadata bilgileriyle ChromaDB koleksiyonuna ekleyerek
# ekleme öncesi ve sonrası koleksiyon boyutunu gösterir.

def add_document_to_collection(ids, metadatas, text_chunksinTokens, chroma_collection):

  # Ekleme işleminden önceki koleksiyon boyutunu yazdır
  print("Ekleme işleminden önceki koleksiyon boyutu: ", chroma_collection.count())

  # Belgeleri, metadata ve ID'leri belirterek ChromaDB koleksiyonuna ekle
  chroma_collection.add(ids=ids, metadatas= metadatas, documents=text_chunksinTokens)

  # Ekleme işleminden sonraki koleksiyon boyutunu yazdır
  print("Ekleme işleminden sonraki koleksiyon boyutu: ", chroma_collection.count())

  # Güncellenmiş koleksiyon nesnesini döndür
  return chroma_collection

## İş Akışını Tasarla

In [21]:
# # load_multiple_pdfs_to_ChromaDB() fonksiyonu, dizindeki tüm PDF'leri okuyup metinleri chunk ve token'lara ayırarak
# metadata ve ID'leriyle birlikte ChromaDB koleksiyonuna ekler.

def load_multiple_pdfs_to_ChromaDB(collection_name,sentence_transformer_model):

  # Koleksiyon adını belirle
  collection_name= collection_name

  # Kategori adını belirle
  category= "Yemek Tarifleri"

  # Kullanılacak Sentence Transformer modelini belirle
  sentence_transformer_model=sentence_transformer_model

  # Embedding fonksiyonunu oluştur
  embedding_function= embedding_functions.SentenceTransformerEmbeddingFunction(model_name=sentence_transformer_model)

  # ChromaDB istemcisi ve koleksiyonunu oluştur veya al
  chroma_client, chroma_collection = create_chroma_client(vector_database_path, collection_name, embedding_function)

  # Mevcut koleksiyonun boyutunu al (ID'leri başlatmak için)
  current_id = chroma_collection.count()

  # Bilgi tabanındaki dosya isimlerini al
  file_names = create_knowledge_base()

  # Her dosya için döngü
  for file_name in file_names:

    # Belge işleme başlangıç mesajı
    print(f"\n📖 {file_name}, {chroma_collection.name} koleksiyonuna eklenmek üzere işleniyor. Kolleksiyondaki mevcut parça sayısı: {chroma_collection.count()}")
    print(f"\n📚 Kolleksiyondaki mevcut parça sayısı: {chroma_collection.count()}")

    # Başlangıç ID'sini yazdır
    print(f"\tBaşlangıç ID: {current_id} ")

    # PDF metnini sayfa sayfa çıkar
    pdf_texts = convert_PDF_Text(file_name)

    # Metni karakter bazlı parçalara ayır
    text_chunksinChar = convert_Page_ChunkinChar(pdf_texts)

    # Karakter bazlı parçaları token bazlı parçalara ayır
    text_chunksinTokens = convert_Chunk_Token(text_chunksinChar,sentence_transformer_model)

    # Metadata ve ID'leri hazırla
    ids,metadatas = add_meta_data(text_chunksinTokens,file_name,category, current_id)

    # Sonraki belge için başlangıç ID'sini güncelle
    current_id = current_id + len(text_chunksinTokens)

    # Belgeleri ChromaDB koleksiyonuna ekle
    chroma_collection = add_document_to_collection(ids, metadatas, text_chunksinTokens, chroma_collection)

    # Belgenin eklendiğine dair mesaj ve güncel parça sayısını yazdır
    print(f"📖{file_name} koleksiyona eklendi. Kolleksiyondaki mevcut parça sayısı: {chroma_collection.count()}\n")

  print(f"📚Koleksiyona belgeler yüklendi👍\n")
  print(f"📚Kolleksiyondaki mevcut parça sayısı: {chroma_collection.count()}\n")
  # ChromaDB istemcisi ve koleksiyonunu döndür
  return  chroma_client, chroma_collection


## İş Akışını Çalıştır

In [22]:
collection_name = "Tarifler"


In [23]:
chroma_client, chroma_collection= load_multiple_pdfs_to_ChromaDB(collection_name,sentence_transformer_model)

'Tarifler' adlı yeni koleksiyon oluşturuldu.
Yerel dizinde bulunan PDF dosyaları:
📄 yemek_tarifleri_rehberi.pdf

Toplam 1 PDF dosyası bulundu.

📖 yemek_tarifleri_rehberi.pdf, Tarifler koleksiyonuna eklenmek üzere işleniyor. Kolleksiyondaki mevcut parça sayısı: 0

📚 Kolleksiyondaki mevcut parça sayısı: 0
	Başlangıç ID: 0 
Belge:  yemek_tarifleri_rehberi.pdf 
Sayfa Sayısı:  15

Toplam parça sayısı (belge maksimum karakter boyutuna göre bölündü = 1500):         22


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]


Belge 128 token'lık parçalara bölündüğünde,
  ve çakışma token sayısı 10 olduğunda
  toplam parça sayısı: 79
Ekleme işleminden önceki koleksiyon boyutu:  0
Ekleme işleminden sonraki koleksiyon boyutu:  79
📖yemek_tarifleri_rehberi.pdf koleksiyona eklendi. Kolleksiyondaki mevcut parça sayısı: 79

📚Koleksiyona belgeler yüklendi👍

📚Kolleksiyondaki mevcut parça sayısı: 79



## Sorgulama yap

In [24]:

sample_queries=[
    "yayla çorbası tarifi verir misin ",
    "etli ve tavuklu ana yemekleri say ",
    "orman kebabı yemeğine hangi malzemeler girer?",
    "yoğurt ile yapılan yemekler nelerdir?",
    "salça kullanılmayan yemekler nelerdir? "
]


In [25]:
import textwrap # Metinleri belirli bir genişliğe göre sarmak için kullanılan kütüphane
from IPython.display import display # Jupyter/Colab ortamında çıktıları daha düzenli göstermek için
from IPython.display import Markdown # Markdown formatında çıktı oluşturmak için

# Normal metni Markdown formatına dönüştüren ve girinti ekleyen fonksiyon
def to_markdown(text):

  # Madde işaretlerini (•) Markdown listesi formatına ( * ) dönüştür
  text = text.replace('•', '  *')

  # Metne her satırın başına '> ' ekleyerek girinti uygula
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))


In [26]:
# ChromaDB koleksiyonundan ilgili belgeleri sorgulamak için fonksiyon
def retrieveDocs(chroma_collection, query, n_results=5, return_only_docs=False):

    # ChromaDB koleksiyonunu sorgula

    # query_texts: Sorgu metinlerinin listesi

    # include: Döndürülecek bilgileri belirtir (belgeler, metadatalar, mesafeler)

    # n_results: Döndürülecek sonuç sayısı (en benzer ilk k belge)
    results = chroma_collection.query(query_texts=[query],
                                      include= [ "documents","metadatas",'distances' ],
                                      n_results=n_results)

    # Eğer sadece belge içeriği isteniyorsa, ilk belge listesini döndür
    if return_only_docs:

        return results['documents'][0]
    # Aksi takdirde, tüm sonuçları (belgeler, metadatalar, mesafeler) içeren sözlüğü döndür

    else:
        return results

In [27]:
def show_results(results, return_only_docs=False):
  output_string = "" # Çıktıyı biriktirmek için boş bir dize değişkeni oluştur

  # Eğer sadece belge içeriği döndürülmüşse (return_only_docs=True)
  if return_only_docs:

    retrieved_documents = results # Sonuçlar doğrudan belge listesidir

    if len(retrieved_documents) == 0: # Boş liste kontrolü
      output_string += "🚩İlgili Sonuç bulunamadı...\n" # Sonuç bulunamadı mesajını dizeye ekle

      return output_string # Dizeyi döndür

    for i, doc in enumerate(retrieved_documents): # Belgeler üzerinde döngü
      output_string += f"Metin Parçası No: {i+1}: " # Belge numarasını dizeye ekle
      output_string += to_markdown(doc).data + "\n" # Belge metnini Markdown formatında alıp dizeye ekle

  # Eğer tam sonuç sözlüğü döndürülmüşse (return_only_docs=False)
  else:
      retrieved_documents = results['documents'][0] # Belge metinlerini al

      if len(retrieved_documents) == 0: # Boş belge listesi kontrolü
          output_string += "🚩İlgili Sonuç bulunamadı...\n" # Sonuç bulunamadı mesajını dizeye ekle

          return output_string # Dizeyi döndür

      retrieved_documents_metadata = results['metadatas'][0] # Metadata bilgisini al
      retrieved_documents_distances = results['distances'][0] # Mesafe bilgisini al


      for i, doc in enumerate(retrieved_documents): # Belgeler üzerinde döngü

          output_string += f"Metin Parçası No: {i+1}: " # Belge numarasını dizeye ekle

          output_string += to_markdown(doc).data + "\n" # Belge metnini Markdown formatında alıp dizeye ekle

          output_string += f"Metin Parçası Kaynağı: {retrieved_documents_metadata[i]['document']}\n" # Belge kaynağını dizeye ekle

          output_string += f"Metin Parçası Kaynak Tipi: {retrieved_documents_metadata[i]['category']}\n" # Belge kategorisini dizeye ekle

          output_string += f"Metin Parçasının Anlamsal Uzaklığı: {retrieved_documents_distances[i]}\n" # Belge mesafesini dizeye ekle

  return output_string # Oluşturulan dizeyi döndür

In [28]:
query_no=2 # Kullanılacak örnek sorgunun listedeki indeksi
print(f"Sorgu: {sample_queries[query_no]}") # Seçilen sorguyu yazdır

# retrieveDocs fonksiyonunu çağırarak ilgili belgeleri al
# chroma_collection: Sorgu yapılacak ChromaDB koleksiyonu

query= sample_queries[query_no]

top_k=3 # En benzer ilk k belgeyi getir

return_only_docs=False #Sadece belge içeriğini değil, tam sonuç sözlüğünü döndür (metadata ve mesafeler dahil)

retrieved_documents=retrieveDocs(chroma_collection,
                                 query,
                                 n_results=top_k,
                                 return_only_docs=return_only_docs)


# show_results fonksiyonunu çağırarak alınan belgeleri göster
chunks=show_results(retrieved_documents, return_only_docs=return_only_docs)
print(chunks)

Sorgu: orman kebabı yemeğine hangi malzemeler girer?
Metin Parçası No: 1: > Geleneksel Türk Mutfağı Yemek Tarifleri Rehberi Çorbalardan Ana Yemeklere, Zeytinyağlılardan Tatlılara 20 Otantik Tarif Bu rehber, Anadolu mutfak kültürünün asırlık geleneklerinden süzülerek günümüze ulaşmış en bilinen ve sevilen tariflerini bir araya getirmektedir. Her bir tarif ; malzeme listesi, aşama aşama hazırlama talimatları ve püf noktaları ile eksiksiz olarak sunulmuştur. Bildirim ve
Metin Parçası Kaynağı: yemek_tarifleri_rehberi.pdf
Metin Parçası Kaynak Tipi: Yemek Tarifleri
Metin Parçasının Anlamsal Uzaklığı: 0.7563711404800415
Metin Parçası No: 2: > Hazırlanan kıymalı harcı patlıcanların ortasına cömertçe doldurun. Üzerlerini domates ve biber dilimleriyle süsleyin. 1 yemek kaşığı salçayı 1 su bardağı sıcak suda eritip tepsinin tabanına dökün. Önceden ısıtılmış 190 derece fırında 25 - 30 dakika pişirin. 7. Geleneksel Orman Kebabı Bol sebzeli ve lokum kıvamında etiyle Bolu ve çevresinden tüm yurda ya


## Sistem Yönlendirmesini Hazırla

In [29]:
system_prompt = """
You are a recipe assistant.

Your task is to provide recipes based only on the information provided in the context.

Rules:
- Answer only recipe-related questions.
- Provide the requested recipe clearly and completely.
- Include the ingredients and preparation steps when available.
- Do not invent or add ingredients, quantities, or preparation steps.
- If the requested recipe is not available in the provided context, say that the recipe is not available.
- Do not use information from your own knowledge when the requested recipe is not in the context.
- Keep your answers clear, concise, and easy to follow.
"""

## 🧠 Bağlam Oluştur

In [30]:

def generate_context_prompt(query, chunks):
  context_prompt = f"""
  ### Kullanıcı Sorgusu:
  {query}

  ### Erişilen Belgeler:
  {chunks}
  """

  print(context_prompt)
  return context_prompt


## Sohbet Nesnesi Oluşturma

In [31]:
from google import genai
from google.genai import types
from IPython.display import Markdown
from google.colab import userdata
# chat nesnesi için client gereki client nesnesi için apı gerekli

# 1. API Anahtarını al ve Client'ı GLOBAL olarak oluştur
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
client = genai.Client(api_key=GOOGLE_API_KEY)

# 2. Chat oluşturma fonksiyonu (client artık global nesneyi kullanır)
def create_chat(system_prompt, model_name):
    chat_config = types.GenerateContentConfig(
        system_instruction=system_prompt,
    )
    # Global istemciden sohbet başlat
    chat = client.chats.create(
        model=model_name,
        config=chat_config,
    )
    return chat



## Cevap Oluşturma

In [32]:
# 3. Yanıt üretme fonksiyonu
def generate_answer(chat, user_prompt):
    response = chat.send_message(user_prompt)
    return response.text

In [33]:
# Bu kod, kullanıcıdan bir yemek tarifi sorusu alır, ChromaDB'den soruyla
# en alakalı tarif parçalarını getirir ve bu parçaları kullanıcı sorusuyla birlikte LLM'e göndererek cevap oluşturur.

from IPython.display import Markdown
model_name = "gemini-2.0-flash"   # --> request sınır aşımı hatası veriyor
# model_name = "gemini-1.5-flash" #  404 not found error

# model_name = "gemini-2.5-flash"  --> 404 not found error
chat = create_chat(system_prompt, model_name)

# Kullanıcıya açıklayıcı karşılama mesajı
print(" Yemek Tarifleri Aistanına Hoşgeldiniz!")
print("Bu bot, yalnızca Türk mutfağına ait lezzetlerin tariflerini verir.")
print("Lütfen tarifini almak istediğiniz yemeği veya herhangi bir tarif hakkında istediğiniz soruyu yazınız.")
print("Çıkmak için 'çık' yazabilirsiniz.\n")

while True:
    # Kullanıcıdan soru al
    print("✏️ Sorunuzu yazınız ('çık' yazarak çıkabilirsiniz): ")
    user_query = input()

    if user_query.lower().strip() == "çık":
        print("\n🔚 Hizmeti kullandığınız için teşekkür ederiz. Başarılar dileriz.")
        break

    # Sorguya göre belgeleri getir
    top_k = 10  # En benzer ilk 3 belgeyi getir
    return_only_docs = False  # Metadata ve mesafe bilgileriyle birlikte döndür
    retrieved_documents = retrieveDocs(
        chroma_collection,
        user_query,
        n_results=top_k,
        return_only_docs=return_only_docs
    )

    # Belgeleri göster
    chunks = show_results(retrieved_documents, return_only_docs=return_only_docs)

    # Kullanıcı sorgusu + erişilen belgelerle bağlam oluştur
    context_prompt = generate_context_prompt(user_query, chunks)

    # LLM ile cevap üret
    llm_answer = generate_answer(chat,context_prompt)

    # Cevabı biçimlendirerek göster
    display(Markdown(f"📥 **Soru:** {user_query}\n\n🧠 **Yanıt:**\n\n{llm_answer}"))

    print("\n---\n")

 Yemek Tarifleri Aistanına Hoşgeldiniz!
Bu bot, yalnızca Türk mutfağına ait lezzetlerin tariflerini verir.
Lütfen tarifini almak istediğiniz yemeği veya herhangi bir tarif hakkında istediğiniz soruyu yazınız.
Çıkmak için 'çık' yazabilirsiniz.

✏️ Sorunuzu yazınız ('çık' yazarak çıkabilirsiniz): 
domates çorbası tarifi verir misin?

  ### Kullanıcı Sorgusu:
  domates çorbası tarifi verir misin?

  ### Erişilen Belgeler:
  Metin Parçası No: 1: > ##mek, bulgur ve pirincin enfes uyumuyla hazırlanan, domates salçası ve nane kokulu klasik bir tencere çorbasıdır. Malzemeler : 1. 5 su bardağı kırmızı mercimek 1 yemek kaşığı pilavlık bulgur 1 yemek kaşığı pirinç 1 adet kuru soğan 2 diş sarımsak 1 yemek kaşığı domates salçası, 1 tatlı kaşığı biber salçası 2 yemek kaşığı tereyağı, 1 yemek kaşığı sıv
Metin Parçası Kaynağı: yemek_tarifleri_rehberi.pdf
Metin Parçası Kaynak Tipi: Yemek Tarifleri
Metin Parçasının Anlamsal Uzaklığı: 0.42865848541259766
Metin Parçası No: 2: > ##mış ) 2 adet yeşil biber,

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.0-flash is no longer available. Please update your code to use models/gemini-3.6-flash for the latest features and improvements.', 'status': 'NOT_FOUND'}}

In [39]:
from IPython.display import Markdown
# model_name = "gemini-2.0-flash"   # --> request sınır aşımı hatası veriyor
# model_name = "gemini-1.5-flash" #  404 not found error
model_name= "models/gemini-3.6-flash"

# model_name = "gemini-2.5-flash"  --> 404 not found error
chat = create_chat(system_prompt, model_name)
user_query="fırında salçalı tavuk incik tarifi verir misin?"

 # Sorguya göre belgeleri getir
top_k = 5  # En benzer ilk 3 belgeyi getir
return_only_docs = False  # Metadata ve mesafe bilgileriyle birlikte döndür
retrieved_documents = retrieveDocs(
        chroma_collection,
        user_query,
        n_results=top_k,
        return_only_docs=return_only_docs
    )

# Belgeleri göster
chunks = show_results(retrieved_documents, return_only_docs=return_only_docs)

# Kullanıcı sorgusu + erişilen belgelerle bağlam oluştur
context_prompt = generate_context_prompt(user_query, chunks)

# LLM ile cevap üret
llm_answer = generate_answer(chat,context_prompt)

# Cevabı biçimlendirerek göster
display(Markdown(f"📥 **Soru:** {user_query}\n\n🧠 **Yanıt:**\n\n{llm_answer}"))


  ### Kullanıcı Sorgusu:
  fırında salçalı tavuk incik tarifi verir misin?

  ### Erişilen Belgeler:
  Metin Parçası No: 1: > 8. Fırında Salçalı Tavuk İncik Patates ve baharatlı sos ile marine edilerek fırında nar gibi kızartılan pratik ve lezzetli bir ana yemektir. Malzemeler : 8 adet tavuk incik veya sarma 3 adet patates ( elma dilim doğranmış ) 1 yemek kaşığı domates salçası, 1 tatlı kaşığı biber salçası 3 yemek kaşığı yoğurt, 3 yemek kaşığı zeytinyağı 3 diş ezilmiş sarımsak 1 tatlı
Metin Parçası Kaynağı: yemek_tarifleri_rehberi.pdf
Metin Parçası Kaynak Tipi: Yemek Tarifleri
Metin Parçasının Anlamsal Uzaklığı: 0.4439082741737366
Metin Parçası No: 2: > ##ın. İmkân varsa 1 saat buzdolabında dinlendirin. Fırın tepsisine dizip önceden ısıtılmış 200 derece fırında tavukların üzeri nar gibi kızarana kadar yaklaşık 40 - 45 dakika pişirin. BÖLÜM 3 : ZEYTİNYAĞLILAR VE SEBZE YEMEKLERİ 9. Zeytinyağlı Yaprak Sarması Ege ve Marmara mutfağının zarif lezzeti ; pirinç, soğan, fı
Metin Parçası Kayn

📥 **Soru:** fırında salçalı tavuk incik tarifi verir misin?

🧠 **Yanıt:**

Belgelerde yer alan bilgilere göre **Fırında Salçalı Tavuk İncik** tarifi şu şekildedir:

### **Malzemeler:**
* 8 adet tavuk incik veya sarma
* 3 adet patates (elma dilim doğranmış)
* 1 yemek kaşığı domates salçası
* 1 tatlı kaşığı biber salçası
* 3 yemek kaşığı yoğurt
* 3 yemek kaşığı zeytinyağı
* 3 diş ezilmiş sarımsak
*(Not: Belgelerde malzeme listesinin devamı kesilmiştir)*

### **Hazırlanışı:**
1. İmkân varsa 1 saat buzdolabında dinlendirin.
2. Fırın tepsisine dizip önceden ısıtılmış 200 derece fırında tavukların üzeri nar gibi kızarana kadar yaklaşık 40 - 45 dakika pişirin.

## Hesabımın erişebildiği modeller

In [35]:
# API anahtarınızın erişebildiği tüm modelleri listeler
for m in client.models.list():
    if "generateContent" in m.supported_actions:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.6-preview
models/gemini-robotics-er-2-preview
models/gemini-2.5-computer-use-p